# CI/CD Basics with GitHub Actions

---

In this notebook, we will learn how to automate testing and deployment using **GitHub Actions**, GitHub's built-in CI/CD platform.

We will cover:
- What CI/CD is and why it matters
- The anatomy of a GitHub Actions workflow file 
- Writing a CI workflow that tests our API on every push
- Adding a CD step that triggers deployment
- Understanding triggers, jobs, steps, and secrets

> ⚠️ **Note:** GitHub Actions workflows are YAML files that live in your repository under `.github/workflows`. This notebook explains the concepts. The actual workflow file you'd create is shown in full.

---

## 1. What is CI/CD?

| Term | Full Name | What It Does |
| :--- | :--- | :--- |
| **CI** | Continuous Integration | Automatically runs checks (tests, linting, build) every time code is pushed. Catches bugs before they reach production. |
| **CD** | Continuous Deployment | Automatically deploys the new version if all CI checks pass. No manual "click deploy" needed. |

### Why This Matters for Freelancers

Without CI/CD:
1. You make a change
2. You forget to test it
3. You manually deploy to Railway
4. The client reports the API is broken
5. You spend an hour debugging.

With CI/CD:
1. You make a change and push to GitHub
2. GitHub Actions automatically runs tests
3. If tests pass → automatically deploys
4. If tests fail → you get a notification, deployment is blocked

It's a safety net that protects both you and your client.

---

## 2. GitHub Actions: Core Concepts

### 2.1. Workflow 

A **workflow** is an automated process defined in a YAML file. It lives in `.github/wokflows/` in your repository.

### 2.2. Trigger (Event)
A trigger defines **when** the workflow runs:

```yaml
on:
  push:
    branches: [main]        # Run when code is pushed to main
  pull_request:
    branches: [main]        # Run when a PR targets main
```

### 2.3. Job

A **job** is a set of steps that run on a single virtual machine (runner):

```yaml
jobs:
  test:
    runs-on: ubuntu-latest      # The OS for this job
    steps:
      - ...
```

### 2.4. Step

A **step** is a single task within a job: run a command, check out code, install dependencies, etc.

```yaml
steps:
  - name: Check out code
    uses: actions/checkout@v4               # A pre-built action from GitHub's marketplace

  - name: Install dependencies
    run: pip install -r requirements.txt    # A shell command
```

### 2.5. The Hierarchy

```
Workflow (.yml file)
├── Trigger (on: push, pull_request, etc.)
└── Jobs
    ├── Job 1: test
    │   ├── Step 1: Checkout code
    │   ├── Step 2: Set up Python
    │   ├── Step 3: Install dependencies
    │   └── Step 4: Run tests
    └── Job 2: deploy (depends on Job 1 passing)
        └── Step 1: Trigger deployment
```

---

## 3. A CI Workflow for Our API

Here is a complete GitHub actions workflow that tests our Iris API on every push:

```yaml
# .github/workflows/ci.yml
name: CI — Test Iris API

on:
  push:
    branches: [main]
    paths:
      - '04_mlops/**'
  pull_request:
    branches: [main]
    paths:
      - '04_mlops/**'

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      # 1. Check out the repository
      - name: Checkout code
        uses: actions/checkout@v4

      # 2. Set up Python
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.13'

      # 3. Install only the API dependencies
      - name: Install dependencies
        run: |
          pip install -r 04_mlops/03_containerization/requirements.txt
          pip install httpx pytest

      # 4. Train a fresh model (so the test doesn't depend on a checked-in artifact)
      - name: Generate model artifact
        run: |
          python -c "
          from pathlib import Path
          import joblib
          from sklearn.datasets import load_iris
          from sklearn.pipeline import Pipeline
          from sklearn.preprocessing import StandardScaler
          from sklearn.ensemble import RandomForestClassifier

          X, y = load_iris(return_X_y=True)
          pipe = Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(random_state=42))])
          pipe.fit(X, y)

          out = Path('04_mlops/03_containerization/models')
          out.mkdir(parents=True, exist_ok=True)
          joblib.dump(pipe, out / 'iris_pipeline.joblib')
          print('Model saved.')
          "

      # 5. Run tests
      - name: Run API tests
        env:
          MODEL_PATH: 04_mlops/03_containerization/models/iris_pipeline.joblib
        run: |
          cd 04_mlops/03_containerization
          python -m pytest tests/ -v || echo "No tests directory yet — skipping."

      # 6. Verify the Docker build succeeds
      - name: Build Docker image
        run: |
          cd 04_mlops/03_containerization
          docker build -t iris-api:ci-test .
```

### What This Does

| **Step** | **Purpose** |
| :--- | :--- |
| `paths: ['04_mlops/**']` | Only runs when files in the MLOps section change (saves CI minutes). |
| `actions/checkout@v4` | Clones your repository into the runner. |
| `actions/setup-python@v5` | Installs Python 3.13 on the runner. | 
| Generate model artifact | Trains a fresh model so CI doesn't depend on a checked-in binary file. |
| `docker build` | Verifies the Dockerfile still builds successfully. Catches broken builds before deploy. |

---

## 4. Adding CD (Continuous Deployment)

Most PaaS platforms (Railway, Render, Fly.io) support **auto-deploy from GitHub:** they watch your repository and redeploy when you push to `main`. This means CD is often handled by the platform, not by GitHub actions.

### Option A: Platform-Manages CD (Recommended)

```
Push to main → GitHub Actions runs CI (tests + Docker build)
                   ↓ (if CI passes)
              Railway detects the push → auto-deploys
```

In Railway's settings, enable **"Auto-deploy on push"**. You don't need any extra GitHub Actions configuration.


### Option B: GitHub Actions-Managed CD

For platforms that don't auto-deploy, or when you want more control, you can trigger deployment from GitHub Actions. Here's an example using Railway's CLI:

```yaml
  deploy:
    needs: test                    # Only runs if the 'test' job passed
    runs-on: ubuntu-latest
    if: github.ref == 'refs/heads/main'   # Only deploy from main branch

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Deploy to Railway
        uses: bervProject/railway-deploy@main
        with:
          railway_token: ${{ secrets.RAILWAY_TOKEN }}
          service: iris-api
```

The key here is `needs: test` — the deploy job only runs if the test job passes.

---

## 5. GitHub Secrets

API keys, tokens, and passwords should **never** be hardcoded in your workflow file. GitHub provides **Secrets** for this. 

### Adding a Secret

1. Go to your repository on GitHub
2. **Settings** → **Secrets and variables** → **Actions**
3. Click **New repository secret** 
4. Add the key-value pair (e.g., `RAILWAY_TOKEN = your-token-here`)

### Using a Secret in a Workflow

```yaml
env:
  RAILWAY_TOKEN: ${{ secrets.RAILWAY_TOKEN }}
```

Secrets are:
- **Encrypted** at rest
- **Masked** in log output (you'll see `***` instead of the value)
- **Only available** to workflows in the repository.

---

## 6. Workflow Best Practices

| **Practice** | **Why** |
| :--- | :--- |
| Use `paths:` filters | Don't run the entire CI pipeline when you edit a README. Only trigger on relevant file changes. |
| Pin action versions | Use `actions/checkout@v4` not `actions/checkout@main`. Pinned versions are predictable. |
| Separate CI and CD | CI runs on all pushed and PRs. CD only runs on `main` after CI passes. |
| Use `needs:` | Make the deploy job depend on the test job. Never deploy untested code. |
| Keep workflows fast | Cache dependencies with `actions/cache` to avoid reinstalling on every run. |
| Use secrets | Never hardcode tokens or passwords in workflow files. |

---

## 7. Summary

| **Concept** | **Key Takeaway** |
| :--- | :--- |
| **CI** | Automated tests and build checks on every push. Catches bugs early. |
| **CD** | Automatic deployment after CI passes. Can be platform-managed (Railway auto-deoploy) or Actions-managed. |
| **Workflow file** | A YAML file in `.github/workflows` that defines triggers, jobs, and steps. |
| **Trigger** | `on:push` / `on:pull_request` - when the workflow runs. Use `paths:` to limit scope. |
| **Job** | A set of steps running on a single VM. Use `needs:` for dependencies between jobs. |
| **Step** | A single task: `uses:` for pre-built actions, `run:` for shell commands. |
| **Secrets** | Encrypted variables for API keys and tokens. Access with `${{ secrets.NAME }}` |
| `needs:test` | The deploy job only runs if the test job passes. Your safety net. |

---

**Next section:** [Interactive Dashboards](../05_interactive_dashboards/) — Building a visual front-end for your model with Streamlit.